In [8]:
"""
==============================================================
HITUNG Z-SCORE & GABUNG DATA STUNTING
==============================================================
Input  1 : stunting.xlsx (JK, Usia, BB, TB) — belum ada ZS
Input  2 : dataset_gabungan.csv (dataset existing)
Input  3 : tabel WHO lhfa boys & girls
Output   : dataset_gabungan.csv (diupdate dengan data baru)
==============================================================
"""

import pandas as pd
import numpy as np

# ──────────────────────────────────────────────
# FASE 1: LOAD DATA STUNTING & TABEL WHO
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 1: LOAD DATA")
print("=" * 55)

df = pd.read_csv('dummy_stunting_300.csv')
who_boys  = pd.read_csv('tab_lhfa_boys_p_0_5.csv')
who_girls = pd.read_csv('tab_lhfa_girls_p_0_5.csv')

print(f"✅ Data stunting dimuat : {len(df)} baris")
print(f"   Kolom              : {df.columns.tolist()}")
print(f"✅ Tabel WHO dimuat\n")


# ──────────────────────────────────────────────
# FASE 2: HITUNG ZS TB/U
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 2: HITUNG ZS TB/U (dari tabel WHO)")
print("=" * 55)

def hitung_zs_tbu(row):
    jk    = str(row['JK']).strip()
    usia  = int(row['Usia'])
    tinggi = float(row['TB'])

    tabel = who_boys if jk == 'L' else who_girls
    who_row = tabel[tabel['Month'] == usia]

    if who_row.empty:
        return np.nan

    M  = who_row['M'].values[0]
    SD = who_row['SD'].values[0]
    return round((tinggi - M) / SD, 2)

df['ZS TB/U'] = df.apply(hitung_zs_tbu, axis=1)
print(f"✅ ZS TB/U berhasil dihitung")
print(f"   Range: {df['ZS TB/U'].min():.2f} s/d {df['ZS TB/U'].max():.2f}")
print(f"   Missing: {df['ZS TB/U'].isna().sum()} baris\n")


# ──────────────────────────────────────────────
# FASE 3: ISI ZS BB/U & ZS BB/TB DENGAN NaN
# (tidak punya tabel WHO untuk ini)
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 3: SET ZS BB/U & ZS BB/TB")
print("=" * 55)

df['ZS BB/U']  = np.nan
df['ZS BB/TB'] = np.nan
print("⚠️  ZS BB/U & ZS BB/TB diset NaN (tidak ada tabel WHO)")
print("   Baris ini akan di-drop saat data_preparation.py\n")


# ──────────────────────────────────────────────
# FASE 4: VALIDASI — SEMUA HARUS SANGAT PENDEK
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 4: VALIDASI LABEL STUNTING")
print("=" * 55)

def cek_status(zs):
    if pd.isna(zs): return 'NaN'
    elif zs < -3:   return 'Sangat Pendek'
    elif zs < -2:   return 'Pendek'
    else:           return 'Normal'

df['_status'] = df['ZS TB/U'].apply(cek_status)
print("Distribusi berdasarkan ZS TB/U:")
print(df['_status'].value_counts().to_string())


# ──────────────────────────────────────────────
# FASE 5: SUSUN KOLOM SESUAI DATASET GABUNGAN
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 5: SUSUN KOLOM")
print("=" * 55)

# Bersihkan JK
df['JK'] = df['JK'].str.strip()

# Rename kolom agar sesuai
df.rename(columns={
    'Usia': 'Usia_Bulan',
    'BB'  : 'Berat',
    'TB'  : 'Tinggi'
}, inplace=True)

# Drop kolom validasi sementara
df.drop(columns=['_status'], inplace=True)

# Susun kolom sesuai dataset gabungan
df_baru = df[['JK', 'Usia_Bulan', 'Berat', 'Tinggi', 'ZS BB/U', 'ZS TB/U', 'ZS BB/TB']].copy()

print(f"✅ Kolom: {df_baru.columns.tolist()}")
print(f"\nSample data:")
print(df_baru.head(3).to_string())
print()


# ──────────────────────────────────────────────
# FASE 6: GABUNGKAN DENGAN DATASET GABUNGAN
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 6: GABUNGKAN DENGAN DATASET GABUNGAN")
print("=" * 55)

df_lama = pd.read_csv('dataset_gabungan5.csv')
print(f"✅ Dataset lama dimuat: {len(df_lama)} baris")

df_gabung = pd.concat([df_lama, df_baru], ignore_index=True)
print(f"✅ Dataset baru       : {len(df_baru)} baris")
print(f"✅ Total gabungan     : {len(df_gabung)} baris\n")


# ──────────────────────────────────────────────
# FASE 7: CEK DUPLIKAT
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 7: CEK DUPLIKAT")
print("=" * 55)

duplikat = df_gabung.duplicated().sum()
print(f"   Duplikat ditemukan: {duplikat}")
if duplikat > 0:
    df_gabung.drop_duplicates(inplace=True)
    print(f"✅ Duplikat dihapus, sisa: {len(df_gabung)} baris")
else:
    print(f"✅ Tidak ada duplikat")
print()


# ──────────────────────────────────────────────
# FASE 8: SIMPAN
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 8: SIMPAN DATASET GABUNGAN")
print("=" * 55)

df_gabung.to_csv('dataset_gabungan5.csv', index=False)
print(f"✅ dataset_gabungan.csv diupdate!")
print(f"   Total: {len(df_gabung)} baris")
print()

# Cek distribusi akhir
def cek_stunting(zs):
    if pd.isna(zs): return 'NaN'
    elif zs < -3:   return 'Sangat Pendek'
    elif zs < -2:   return 'Pendek'
    else:           return 'Normal'

df_gabung['_cek'] = df_gabung['ZS TB/U'].apply(cek_stunting)
print("Distribusi akhir dataset gabungan:")
for label, count in df_gabung['_cek'].value_counts().items():
    pct = count / len(df_gabung) * 100
    print(f"   {label:<15}: {count:>4} ({pct:.1f}%)")

print()
print("=" * 55)
print("✅ SELESAI — Lanjut jalankan data_preparation.py")
print("=" * 55)

FASE 1: LOAD DATA
✅ Data stunting dimuat : 300 baris
   Kolom              : ['JK', 'Usia', 'BB', 'TB']
✅ Tabel WHO dimuat

FASE 2: HITUNG ZS TB/U (dari tabel WHO)
✅ ZS TB/U berhasil dihitung
   Range: -6.06 s/d -1.78
   Missing: 0 baris

FASE 3: SET ZS BB/U & ZS BB/TB
⚠️  ZS BB/U & ZS BB/TB diset NaN (tidak ada tabel WHO)
   Baris ini akan di-drop saat data_preparation.py

FASE 4: VALIDASI LABEL STUNTING
Distribusi berdasarkan ZS TB/U:
_status
Sangat Pendek    161
Pendek           131
Normal             8
FASE 5: SUSUN KOLOM
✅ Kolom: ['JK', 'Usia_Bulan', 'Berat', 'Tinggi', 'ZS BB/U', 'ZS TB/U', 'ZS BB/TB']

Sample data:
  JK  Usia_Bulan  Berat  Tinggi  ZS BB/U  ZS TB/U  ZS BB/TB
0  P          28    7.2    76.4      NaN    -3.70       NaN
1  L          19    5.8    71.9      NaN    -4.12       NaN
2  P          15    5.5    66.7      NaN    -3.95       NaN

FASE 6: GABUNGKAN DENGAN DATASET GABUNGAN
✅ Dataset lama dimuat: 3325 baris
✅ Dataset baru       : 300 baris
✅ Total gabungan     

In [3]:
df = pd.read_csv('dataset_gabungan5.csv')

def cek_stunting(zs):
    if pd.isna(zs): return 'NaN'
    elif zs < -3: return 'Sangat Pendek'
    elif zs < -2: return 'Pendek'
    else: return 'Normal'

df['_status'] = df['ZS TB/U'].apply(cek_stunting)
print('Total:', len(df))
print(df['_status'].value_counts())
print()
print('Missing ZS BB/U :', df['ZS BB/U'].isna().sum())
print('Missing ZS BB/TB:', df['ZS BB/TB'].isna().sum())

Total: 3325
_status
Normal           1779
Pendek            765
Sangat Pendek     719
NaN                62
Name: count, dtype: int64

Missing ZS BB/U : 0
Missing ZS BB/TB: 0


In [ ]:
import pandas as pd
import numpy as np

# ──────────────────────────────────────────────
# FASE 1: LOAD DATA STUNTING & TABEL WHO
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 1: LOAD DATA")
print("=" * 55)

df = pd.read_csv('dataset_gabungan5.csv')
who_boys_bb  = pd.read_csv('wfa_boys_p.csv')
who_girls_bb = pd.read_csv('tab_lhfa_girls_p_0_5.csv')

print(f"✅ Data stunting dimuat : {len(df)} baris")
print(f"   Kolom              : {df.columns.tolist()}")
print(f"✅ Tabel WHO dimuat\n")


# ──────────────────────────────────────────────
# FASE 2: HITUNG ZS TB/U
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 2: HITUNG ZS TB/U (dari tabel WHO)")
print("=" * 55)

def hitung_zs_tbu(row):
    jk    = str(row['JK']).strip()
    usia  = int(row['Usia'])
    tinggi = float(row['TB'])

    tabel = who_boys if jk == 'L' else who_girls
    who_row = tabel[tabel['Month'] == usia]

    if who_row.empty:
        return np.nan

    M  = who_row['M'].values[0]
    SD = who_row['SD'].values[0]
    return round((tinggi - M) / SD, 2)

df['ZS TB/U'] = df.apply(hitung_zs_tbu, axis=1)
print(f"✅ ZS TB/U berhasil dihitung")
print(f"   Range: {df['ZS TB/U'].min():.2f} s/d {df['ZS TB/U'].max():.2f}")
print(f"   Missing: {df['ZS TB/U'].isna().sum()} baris\n")


In [8]:
import pandas as pd
import numpy as np

# ──────────────────────────────────────────────
# LOAD DATA
# ──────────────────────────────────────────────
df = pd.read_csv('dummy_sangat_pendek.csv')

# 🔥 RAPIIKAN NAMA KOLOM (WAJIB)
df.columns = df.columns.str.strip().str.lower()

print(f"Dataset dimuat: {len(df)} baris")
print("Kolom tersedia:", df.columns.tolist(), "\n")

# Mapping kolom biar aman
col_jk   = 'jk'
col_usia = 'usia'   # ✅ sesuai dataset
col_bb   = 'bb'     # ✅ sesuai dataset
col_tb   = 'tb'     # ✅ sesuai dataset

# ──────────────────────────────────────────────
# BUAT KOLOM ZS JIKA BELUM ADA
# ──────────────────────────────────────────────
cols_zs = ['zs tb/u', 'zs bb/u', 'zs bb/tb']

for col in cols_zs:
    if col not in df.columns:
        df[col] = np.nan

# ──────────────────────────────────────────────
# LOAD TABEL WHO
# ──────────────────────────────────────────────
who_boys  = pd.read_csv('tab_lhfa_boys_p_0_5.csv')
who_girls = pd.read_csv('tab_lhfa_girls_p_0_5.csv')

wfa_boys  = pd.read_excel('wfa_boys_0-to-5-years_zscores (1).xlsx')
wfa_girls = pd.read_excel('wfa_girls_0-to-5-years_zscores (1).xlsx')

wfh_boys  = pd.read_excel('wfh_boys_2-to-5-years_zscores.xlsx')
wfh_girls = pd.read_excel('wfh_girls_2-to-5-years_zscores.xlsx')
wfl_boys  = pd.read_excel('wfl_boys_0-to-2-years_zscores.xlsx')
wfl_girls = pd.read_excel('wfl_girls_0-to-2-years_zscores.xlsx')

print("✅ Semua tabel WHO dimuat!")

# ──────────────────────────────────────────────
# HITUNG ZS TB/U
# ──────────────────────────────────────────────
def hitung_zs_tbu(row):
    try:
        jk     = str(row[col_jk]).strip()
        usia   = int(row[col_usia])
        tinggi = float(row[col_tb])

        tabel = who_boys if jk == 'L' else who_girls
        who_row = tabel[tabel['Month'] == usia]

        if who_row.empty:
            return np.nan

        M  = who_row['M'].values[0]
        SD = who_row['SD'].values[0]
        return round((tinggi - M) / SD, 2)
    except:
        return np.nan

df['zs tb/u'] = df.apply(hitung_zs_tbu, axis=1)

# ──────────────────────────────────────────────
# KLASIFIKASI STUNTING
# ──────────────────────────────────────────────
def klasifikasi_stunting(z):
    if pd.isna(z):
        return 'Tidak diketahui'
    elif z < -3:
        return 'Sangat Pendek'
    elif z < -2:
        return 'Pendek'
    else:
        return 'Normal'

df['status stunting'] = df['zs tb/u'].apply(klasifikasi_stunting)

# ──────────────────────────────────────────────
# ZS BB/U
# ──────────────────────────────────────────────
def hitung_zs_bbu(jk, usia_bulan, berat):
    try:
        tabel = wfa_boys if jk == 'L' else wfa_girls
        row = tabel[tabel['Month'] == int(usia_bulan)]

        if row.empty:
            return np.nan

        M, S, L = row[['M','S','L']].values[0]

        if L != 0:
            zs = ((berat / M) ** L - 1) / (L * S)
        else:
            zs = np.log(berat / M) / S

        return round(zs, 2)
    except:
        return np.nan

# ──────────────────────────────────────────────
# ZS BB/TB
# ──────────────────────────────────────────────
def hitung_zs_bbtb(jk, tinggi, berat):
    try:
        if tinggi < 65:
            tabel = wfl_boys if jk == 'L' else wfl_girls
            col   = 'Length'
        else:
            tabel = wfh_boys if jk == 'L' else wfh_girls
            col   = 'Height'

        tabel['_diff'] = abs(tabel[col] - tinggi)
        row = tabel.loc[tabel['_diff'].idxmin()]

        M, S, L = row[['M','S','L']]

        if L != 0:
            zs = ((berat / M) ** L - 1) / (L * S)
        else:
            zs = np.log(berat / M) / S

        return round(zs, 2)
    except:
        return np.nan

# ──────────────────────────────────────────────
# DEFINISI NAMA KOLOM ZS (WAJIB)
# ──────────────────────────────────────────────
col_zs_bbu  = 'zs bb/u'
col_zs_bbtb = 'zs bb/tb'

# ──────────────────────────────────────────────
# HITUNG YANG MISSING SAJA
# ──────────────────────────────────────────────
mask_bbu = df[col_zs_bbu].isna()
df.loc[mask_bbu, col_zs_bbu] = df[mask_bbu].apply(
    lambda row: hitung_zs_bbu(row[col_jk], row[col_usia], row[col_bb]), axis=1
)

mask_bbtb = df[col_zs_bbtb].isna()
df.loc[mask_bbtb, col_zs_bbtb] = df[mask_bbtb].apply(
    lambda row: hitung_zs_bbtb(row[col_jk], row[col_tb], row[col_bb]), axis=1
)

# ──────────────────────────────────────────────
# HASIL
# ──────────────────────────────────────────────
print("\nSetelah perhitungan:")
print(f"ZS BB/U : {df[col_zs_bbu].isna().sum()}")
print(f"ZS BB/TB: {df[col_zs_bbtb].isna().sum()}")

# 🔥 SIMPAN KE FILE BARU (AMAN)
df.to_csv('dummy_sangat_pendek.csv', index=False)

print("\n✅ Selesai! File: dummy_sangat_pendek.csv di update")

Dataset dimuat: 100 baris
Kolom tersedia: ['jk', 'usia', 'bb', 'tb'] 

✅ Semua tabel WHO dimuat!

Setelah perhitungan:
ZS BB/U : 0
ZS BB/TB: 0

✅ Selesai! File: dummy_sangat_pendek.csv di update


In [12]:
import pandas as pd

# ─────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────
df1 = pd.read_csv('gabungan_stunting.csv')  # yang ada status stunting
df2 = pd.read_csv('dataset_gabungan2.csv')           # yang tanpa status

# ─────────────────────────────────────
# RAPIIKAN NAMA KOLOM (biar konsisten)
# ─────────────────────────────────────
df1.columns = df1.columns.str.strip().str.lower()
df2.columns = df2.columns.str.strip().str.lower()

# ─────────────────────────────────────
# HAPUS KOLOM YANG TIDAK DIPERLUKAN
# ─────────────────────────────────────
if 'status stunting' in df1.columns:
    df1 = df1.drop(columns=['status stunting'])

# ─────────────────────────────────────
# SAMAKAN NAMA KOLOM
# ─────────────────────────────────────
df1 = df1.rename(columns={
    'jk': 'jk',
    'usia': 'usia_bulan',
    'bb': 'berat',
    'tb': 'tinggi'
})

df2 = df2.rename(columns={
    'jk': 'jk',
    'berat': 'berat',
    'tinggi': 'tinggi',
    'usia_bulan': 'usia_bulan'
})

# ─────────────────────────────────────
# SAMAKAN URUTAN KOLOM
# ─────────────────────────────────────
kolom_final = ['jk', 'usia_bulan', 'berat', 'tinggi',
               'zs tb/u', 'zs bb/u', 'zs bb/tb']

df1 = df1[kolom_final]
df2 = df2[kolom_final]

# ─────────────────────────────────────
# GABUNG DATA
# ─────────────────────────────────────
df = pd.concat([df1, df2], ignore_index=True)

# ─────────────────────────────────────
# SIMPAN
# ─────────────────────────────────────
df.to_csv('bismillah_final.csv', index=False)

print("✅ Dataset berhasil digabung & dibersihkan!")
print(f"Total data: {len(df)} baris")

✅ Dataset berhasil digabung & dibersihkan!
Total data: 2881 baris
